## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [1]:
from dotenv import load_dotenv
from langchain_community.embeddings import HuggingFaceEmbeddings
#from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [2]:
MODEL = "gpt-oss:20b" #"gpt-4.1-nano"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [3]:
#embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings = HuggingFaceEmbeddings(model_name="Qwen/Qwen3-Embedding-8B")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [4]:
retriever = vectorstore.as_retriever()
#llm = ChatOpenAI(temperature=0, model_name=MODEL)
llm = ChatOllama(temperature=0, model=MODEL)

### These LangChain objects implement the method `invoke()`

In [5]:
retriever.invoke("What is the compensation history for Alex Chen?")

[Document(id='68782f64-e2fa-44fe-bdcf-571a04bbe52c', metadata={'source': 'knowledge-base/employees/Avery Lancaster.md', 'doc_type': 'employees'}, page_content='## Compensation History\n- **2015**: $150,000 base salary + Significant equity stake  \n- **2016**: $160,000 base salary + Equity increase  \n- **2017**: $150,000 base salary + Decrease in bonus due to performance  \n- **2018**: $180,000 base salary + performance bonus of $30,000  \n- **2019**: $185,000 base salary + market adjustment + $5,000 bonus  \n- **2020**: $170,000 base salary (temporary reduction due to COVID-19)  \n- **2021**: $200,000 base salary + performance bonus of $50,000  \n- **2022**: $210,000 base salary + retention bonus  \n- **2023**: $225,000 base salary + $75,000 performance bonus'),
 Document(id='d326cb95-1aa0-4618-b34a-ce9977c8ea70', metadata={'doc_type': 'employees', 'source': 'knowledge-base/employees/Alex Chen.md'}, page_content='Alex Chen continues to be a vital asset at Insurellm, contributing signi

In [6]:
llm.invoke("What is the compensation history for Alex Chen?")

AIMessage(content='I’m sorry, but I can’t help with that.', additional_kwargs={}, response_metadata={'model': 'gpt-oss:20b', 'created_at': '2025-11-29T17:45:49.960703Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1202965417, 'load_duration': 117150584, 'prompt_eval_count': 76, 'prompt_eval_duration': 232604084, 'eval_count': 69, 'eval_duration': 833410007, 'model_name': 'gpt-oss:20b', 'model_provider': 'ollama'}, id='lc_run--b0baaeba-f5b8-44de-add3-6c077be924e7-0', usage_metadata={'input_tokens': 76, 'output_tokens': 69, 'total_tokens': 145})

## Time to put this together!

In [8]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [9]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    #print(response)
    return response.content

In [11]:
from IPython.display import display, Markdown

answer = answer_question("What is the compensation history for Brandon Walker?", [])
display(Markdown(answer))

**Brandon Walker – Compensation History (as recorded in the HR file)**  

| Year | Base Salary | Bonus | Total Compensation |
|------|-------------|-------|--------------------|
| **2023** | $105,000 | $18,000 | $123,000 |
| **2022** | $98,000  | $15,000 | $113,000 |
| **2021** | $92,000  | $12,000 | $104,000 |
| **2020** | $88,000  | $8,000  | $96,000 |
| **2019** | $85,000  | $10,000 | $95,000 |

---

### A note on the data

The HR record actually contains **two separate compensation tables** for Brandon Walker, which appear to reflect different roles or reporting periods:

| Year | Base Salary | Bonus | Total Compensation |
|------|-------------|-------|--------------------|
| **2023** | $135,000 | $20,000 | $155,000 |
| **2022** | $128,000 | $15,000 | $143,000 |
| **2021** | $120,000 | $18,000 | $138,000 |
| **2020** | $110,000 | $8,000  | $118,000 |
| **2019** | $105,000 | $10,000 | $115,000 |

The presence of two sets suggests either a data entry overlap or a transition between roles (e.g., moving from a support‑focused position to a higher‑level product or engineering role). The current salary listed in the summary ($62,000) also does not match either table, which may indicate a recent change in position or a discrepancy in the records.

If you need clarification on which table applies to Brandon’s current role or if you’d like to reconcile the figures, let me know and I can help dig into the details.

## What could possibly come next? 😂

In [12]:
gr.ChatInterface(answer_question).launch(inbrowser=True)

/Users/darthjedi/development/udemy/llm_engineering/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## Admit it - you thought RAG would be more complicated than that!!

In [13]:
answer = answer_question("List the compensation history for Jessica Liu.", [])
display(Markdown(answer))

KeyboardInterrupt: 